In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn 

df = pd.read_csv("https://raw.githubusercontent.com/joyceximenes/assistente_IA/main/analysis/data/guidance_scores.csv")

guidance_df = df[df.split == "train"].copy()
guidance_df.shape

In [ ]:
blr_true = (guidance_df.BLR == 1).sum()
prop_blr = blr_true / guidance_df.shape[0]

print(f"Quantidade de positivos (desfocados): {blr_true}, Proporção: {prop_blr:.2%}")

a proporção de 40% indica que aleatoriamente, sem considerar o blurScore, o app acertaria que 4 a cada 10 fotos estão desfocadas

In [ ]:
# quantos alertas gera
desf = (guidance_df.blurScore < 120).sum()
print(f"quantidade de alertas: {desf}")

In [ ]:
# quantos estão de fato desfocados
desf_true = ((guidance_df.blurScore < 120) & (guidance_df.BLR == 1)).sum()
print(f"quantidade de alertas verdadeiros : {desf_true}")

In [ ]:
y_pred = guidance_df["blurScore"] < 120  
y_true = guidance_df["BLR"] == 1          

comparative_blr = sklearn.metrics.confusion_matrix(
    y_true,
    y_pred)

sklearn.metrics.ConfusionMatrixDisplay(comparative_blr,).plot()

In [ ]:
presc_blr = sklearn.metrics.precision_score(y_true, y_pred)
recall_blr = sklearn.metrics.recall_score(y_true, y_pred)


print(f"precisão: {presc_blr:.2%}, revocação: {recall_blr:.2%}")

verificando precisão e recall para diversos cortes de blurScore

In [ ]:
precision_blr, recall_blr, thresholds_pr_blr = sklearn.metrics.precision_recall_curve(y_true, -guidance_df.blurScore)
graph_curve_blr = sklearn.metrics.PrecisionRecallDisplay(precision=precision_blr, recall=recall_blr).plot()

 blurScore sozinho tem um teto de precisão em torno de 76%. então é o maximo que um corte pode dar de precisão

---
DRK/escuro

In [ ]:
drk_true = (guidance_df.DRK == 1).sum()
prop_drk = drk_true / guidance_df.shape[0]

dark_true = ((guidance_df.brightnessScore < 30) & (guidance_df.DRK == 1)).sum()

print(f"Quantidade de positivos (escuras): {drk_true}, Proporção: {prop_drk:.2%}")
print(f"quantidade de alertas verdadeiros : {dark_true}")

In [ ]:
y_pred_drk = guidance_df["brightnessScore"] < 30  
y_true_drk = guidance_df["DRK"] == 1       

presc_drk = sklearn.metrics.precision_score(y_true_drk, y_pred_drk)
recall_drk = sklearn.metrics.recall_score(y_true_drk, y_pred_drk)


print(f"precisão: {presc_drk:.2%}, revocação: {recall_drk:.2%}")

In [ ]:
precision_drk, recall_drk, thresholds_pr_drk = sklearn.metrics.precision_recall_curve(y_true_drk, -guidance_df.brightnessScore)
graph_curve_drk = sklearn.metrics.PrecisionRecallDisplay(precision=precision_drk, recall=recall_drk).plot()

## F-beta: alinhar precisão e revocação com precisão tendo mais peso 

### BLR

verificar para todos os cortes, considerando precisão mais importante que revocação

In [ ]:
import numpy as np

beta = 0.5

numerador = (1 + beta**2) * precision_blr * recall_blr
denominador = beta**2 * precision_blr + recall_blr

f_beta_blr = np.divide(
    numerador,
    denominador,
    out=np.zeros_like(np.asarray(denominador, dtype=float)),
    where=np.asarray(denominador) != 0
)

f_beta_blr.shape

In [ ]:
# 1. Posição do maior F0,5, excluindo o último ponto
idx_best_blr = np.nanargmax(f_beta_blr[:-1])

# 2. Limiar correspondente, convertendo de -blurScore para blurScore
best_threshold_blr = -thresholds_pr_blr[idx_best_blr]

# 3. Precisão e revocação nesse ponto
best_precision_blr = precision_blr[idx_best_blr]
best_recall_blr = recall_blr[idx_best_blr]
best_f_beta_blr = f_beta_blr[idx_best_blr]

print(f"Posição: {idx_best_blr}")
print(f"Corte de blurScore: {best_threshold_blr:.2f}")
print(f"F0,5 máximo: {best_f_beta_blr:.4f}")
print(f"Precisão: {best_precision_blr:.4f}")
print(f"Revocação: {best_recall_blr:.4f}")

In [ ]:

guidance_validation = df[df.split == "val"].copy()

guidance_validation["blr_calibration"] = guidance_validation.blurScore < 123.57

# precisão
presc_blr = sklearn.metrics.precision_score(guidance_validation.BLR, guidance_validation.blr_calibration)

# revocação
rev_blr = sklearn.metrics.recall_score(guidance_validation.BLR, guidance_validation.blr_calibration)

print(f"precisão: {presc_blr:.2%}, revocação: {rev_blr:.2%}")

linha de base: 

precisão 0,712, com IC de 95% de [0,695; 0,729].

revocação 0,592, com IC de 95% de [0,575; 0,609].

ou seja, o corte de 123,57 está dentro da margem de erro calculada. convem deixar 120

## Função: calibrar o corte de qualquer categoria


**Recebe:** rótulo e score do treino, rótulo e score da validação, o corte de hoje, o `sentido` da regra e o `beta` (padrão 0,5). `"menor"`: o app alerta quando o score fica abaixo do corte (BLR e DRK). `"maior"`: alerta quando fica acima (BRT).

**Devolve:** uma tabela com o corte de hoje e o calibrado, lado a lado, com precisão, revocação e F0,5 no treino e na validação.

In [ ]:
import numpy as np
import pandas as pd
import sklearn.metrics


def calibrar_corte(y_treino, score_treino, y_val, score_val, corte_hoje, sentido, beta=0.5):
    # o sklearn assume "maior score = mais provável de ser positivo"; no sentido "menor" inverte o sinal
    sinal = -1 if sentido == "menor" else 1

    precisao, revocacao, limiares = sklearn.metrics.precision_recall_curve(
        y_treino, sinal * score_treino
    )

    # F-beta em cada ponto da curva (0 onde a conta daria 0/0)
    numerador = (1 + beta**2) * precisao * revocacao
    denominador = beta**2 * precisao + revocacao
    f_beta = np.divide(numerador, denominador, out=np.zeros_like(denominador), where=denominador != 0)

    # o último ponto da curva é artificial (não tem limiar): fica fora da busca
    melhor = np.nanargmax(f_beta[:-1])
    corte_novo = sinal * limiares[melhor]

    linhas = {}
    for nome, corte in [("hoje", corte_hoje), ("calibrado", corte_novo)]:
        linha = {"corte": corte}
        for conjunto, y, score in [("treino", y_treino, score_treino), ("validação", y_val, score_val)]:
            alerta = score < corte if sentido == "menor" else score > corte
            linha[f"precisão ({conjunto})"] = sklearn.metrics.precision_score(y, alerta)
            linha[f"revocação ({conjunto})"] = sklearn.metrics.recall_score(y, alerta)
            linha[f"F{beta} ({conjunto})"] = sklearn.metrics.fbeta_score(y, alerta, beta=beta)
        linhas[nome] = linha
    return pd.DataFrame(linhas).T

### Conferência com o BLR

O resultado precisa bater com o que foi feito à mão: corte calibrado ≈ 123,57; na validação, precisão 70,82% e revocação 59,87% (corte de hoje: 71,22% e 59,16%).

In [ ]:
resultado_blr = calibrar_corte(
    guidance_df.BLR == 1, guidance_df.blurScore,
    guidance_validation.BLR == 1, guidance_validation.blurScore,
    corte_hoje=120, sentido="menor",
)
resultado_blr.round(4)

## DRK, BRT

In [ ]:
resultado_drk = calibrar_corte(
    guidance_df.DRK == 1, guidance_df.brightnessScore,
    guidance_validation.DRK == 1, guidance_validation.brightnessScore,
    corte_hoje=30, sentido="menor",
)
resultado_drk.round(4)

linha de base:

Precisão: 0,893, com intervalo de [0,840; 0,942].

Revocação: 0,282, com intervalo de [0,239; 0,324].

precisão e revocação dentro da margem de erro.

In [ ]:
resultado_brt = calibrar_corte(
    guidance_df.BRT == 1, guidance_df.overexposedRatio,
    guidance_validation.BRT == 1, guidance_validation.overexposedRatio,
    corte_hoje=0.15, sentido="maior",
)
resultado_brt.round(4)

linha de base: 

precisão 0,482 [0,423; 0,541] 

revocação 0,288 [0,247; 0,328].

precisão acima da margem de erro e revocação abaixo da margem de erro.

#### conclusão:  só o BRT muda de corte, de 0,15 para 0,22. BLR e DRK mantêm o corte chutado